# Generación de constancias de posters

Este notebook lee `posters.csv`, reemplaza los marcadores `{{NOMBRE}}` y `{{TITULO}}` en `plantilla_constancia.pptx`, y genera un PPTX por registro en `Constancias/Posters/salida_pptx/`.

Requisitos:
- `plantilla_constancia.pptx` en la misma carpeta del notebook o en `Constancias/Plantilla Constancias/Posters/plantilla_constancia.pptx`.
- `posters.csv` en la misma carpeta del notebook o en `Posters/posters.csv`.
- Dependencias: `python-pptx` y `pandas`.
- Para exportar a PDF en Windows: `comtypes` y Microsoft PowerPoint instalado.

Instalación (si hace falta):
```bash
pip install python-pptx pandas
```

In [3]:
from __future__ import annotations

import re
import sys
from pathlib import Path

import pandas as pd
from pptx import Presentation
from pptx.enum.shapes import MSO_SHAPE_TYPE

INVALID_FILENAME_CHARS = r"[\\/:*?\"<>|]"
EXPORTAR_PDF = True


def sanitize_filename(nombre: str, index: int, max_len: int = 120) -> str:
    base = f"{index:02d} _Constancia_Poster_{nombre}".strip()
    base = re.sub(INVALID_FILENAME_CHARS, "", base).strip().strip(".")
    if not base:
        base = f"{index:02d} _Constancia_Poster"
    max_base_len = max_len - len(".pptx")
    if len(base) > max_base_len:
        base = base[:max_base_len].rstrip().rstrip(".")
    return f"{base}.pptx"


def replace_in_paragraph(paragraph, replacements: dict[str, str]) -> bool:
    if not paragraph.runs:
        return False

    original_text = "".join(run.text for run in paragraph.runs)
    updated_text = original_text
    for key, value in replacements.items():
        updated_text = updated_text.replace(key, value)

    if updated_text == original_text:
        return False

    for run in paragraph.runs:
        run_text = run.text
        new_text = run_text
        for key, value in replacements.items():
            new_text = new_text.replace(key, value)
        if new_text != run_text:
            run.text = new_text

    combined = "".join(run.text for run in paragraph.runs)
    if any(key in combined for key in replacements.keys()):
        paragraph.text = updated_text

    return True


def replace_in_text_frame(text_frame, replacements: dict[str, str]) -> bool:
    changed = False
    for paragraph in text_frame.paragraphs:
        if replace_in_paragraph(paragraph, replacements):
            changed = True
    return changed


def iter_shapes(shapes):
    for shape in shapes:
        if shape.shape_type == MSO_SHAPE_TYPE.GROUP:
            for subshape in iter_shapes(shape.shapes):
                yield subshape
        else:
            yield shape


def replace_in_slide(slide, replacements: dict[str, str]) -> None:
    for shape in iter_shapes(slide.shapes):
        if shape.has_text_frame:
            replace_in_text_frame(shape.text_frame, replacements)
        if shape.has_table:
            for row in shape.table.rows:
                for cell in row.cells:
                    replace_in_text_frame(cell.text_frame, replacements)


def exportar_pptx_a_pdf(pptx_path: Path, pdf_path: Path) -> None:
    try:
        import comtypes.client
    except ModuleNotFoundError as exc:
        raise ModuleNotFoundError(
            "No se encontro 'comtypes'. Instala con: pip install comtypes"
        ) from exc

    powerpoint = comtypes.client.CreateObject("PowerPoint.Application")
    powerpoint.Visible = 1
    presentation = powerpoint.Presentations.Open(str(pptx_path), WithWindow=False)
    try:
        presentation.SaveAs(str(pdf_path), 32)  # 32 = ppSaveAsPDF
    finally:
        presentation.Close()
        powerpoint.Quit()


In [4]:
base_dir = Path(".").resolve()
possible_csv_paths = [
    base_dir / "posters.csv",
    base_dir / "Posters" / "posters.csv",
]

possible_template_paths = [
    base_dir / "plantilla_constancia.pptx",
    base_dir / "Constancias" / "Plantilla Constancias" / "Posters" / "plantilla_constancia.pptx",
]
output_dir = base_dir / "Constancias" / "Posters" / "salida_pptx"

csv_path = next((path for path in possible_csv_paths if path.exists()), None)
template_path = next((path for path in possible_template_paths if path.exists()), None)
if csv_path is None:
    raise FileNotFoundError(
        "No se encontro el archivo CSV en: " + ", ".join(str(p) for p in possible_csv_paths)
    )
if template_path is None:
    raise FileNotFoundError(
        "No se encontro la plantilla PPTX en: "
        + ", ".join(str(p) for p in possible_template_paths)
    )

output_dir.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(csv_path, encoding="utf-8", dtype=str, keep_default_na=False)
if "NOMBRE" not in df.columns or "TITULO" not in df.columns:
    print("El CSV debe contener las columnas NOMBRE y TITULO.")
    sys.exit(1)

total = 0
pdf_total = 0
pdf_dir = output_dir / "pdf"
exportar_pdf_activo = EXPORTAR_PDF
if EXPORTAR_PDF:
    pdf_dir.mkdir(parents=True, exist_ok=True)

for index, row in df.iterrows():
    nombre = str(row.get("NOMBRE", "")).strip()
    titulo = str(row.get("TITULO", "")).strip()
    replacements = {
        "{{NOMBRE}}": nombre,
        "{{TITULO}}": titulo,
    }

    presentation = Presentation(template_path)
    for slide in presentation.slides:
        replace_in_slide(slide, replacements)

    filename = sanitize_filename(nombre, index + 1)
    output_path = output_dir / filename
    presentation.save(output_path)
    total += 1

    if exportar_pdf_activo:
        pdf_path = pdf_dir / Path(filename).with_suffix(".pdf").name
        try:
            exportar_pptx_a_pdf(output_path, pdf_path)
            pdf_total += 1
        except ModuleNotFoundError as exc:
            print(str(exc))
            print("Desactivando exportacion a PDF para esta ejecucion.")
            exportar_pdf_activo = False

print(f"Generadas {total} constancias en: {output_dir.resolve()}")
if EXPORTAR_PDF:
    if exportar_pdf_activo:
        print(f"PDFs generados: {pdf_total} en {pdf_dir.resolve()}")
    else:
        print("PDFs no generados: falta 'comtypes' o PowerPoint.")

Generadas 20 constancias en: D:\Repositorios_Git\Semana_Mecatronica_WEB\Constancias\Posters\salida_pptx
PDFs generados: 20 en D:\Repositorios_Git\Semana_Mecatronica_WEB\Constancias\Posters\salida_pptx\pdf
